# Compare Implementation of DML IRM in Causalis and DML IRM in DoubleML

This notebook presents the **doubleml benchmark** research workflow and key analysis steps.

Comparing `IRM` model from Causalis with `dml.DoubleMLIRM` from DoubleML with default CatboostRegressor and CatboostClassifier for g0, g1 amd m

# DGP

We will use DGP: `generate_obs_hte_26_rich()`
read more at [this notebook](https://causalis.causalcraft.com/articles/generate_obs_hte_26_rich)

In [1]:
from causalis.scenarios.unconfoundedness.dgp import generate_obs_hte_26_rich

data = generate_obs_hte_26_rich(return_causal_data=False, include_oracle=True)
data.head()


,user_id,y,d,tenure_months,avg_sessions_week,spend_last_month,age_years,income_monthly,prior_purchases_12m,support_tickets_90d,premium_user,mobile_user,urban_resident,referred_user,m,m_obs,tau_link,g0,g1,cate
0,1,0.000000,0.0,28.814654,1.0,77.936767,50.234101,1926.698301,1.0,2.0,1.0,1.0,1.0,0.0,0.045453,0.045453,0.089095,8.137981,9.142395,1.004414
1,2,80.099611,1.0,25.913345,3.0,53.777740,28.115859,5104.271509,3.0,0.0,1.0,1.0,0.0,1.0,0.041514,0.041514,0.246679,60.459257,78.817307,18.358049
2,3,6.400482,1.0,24.969929,10.0,134.764322,22.907062,5267.938255,8.0,3.0,0.0,1.0,1.0,0.0,0.052593,0.052593,0.162968,7.712855,9.138577,1.425723
3,4,2.788238,0.0,40.655089,5.0,59.517074,31.970490,6597.327018,3.0,2.0,1.0,1.0,1.0,0.0,0.036221,0.036221,0.188755,25.386510,31.159932,5.773422
4,5,0.000000,0.0,18.560899,3.0,74.370930,39.237248,4930.009628,5.0,1.0,1.0,1.0,0.0,0.0,0.036343,0.036343,0.174757,15.359250,18.600227,3.240977


In [2]:
true_atte = data.loc[data["d"] == 1, "cate"].mean()
treated_n = int(data["d"].sum())
treated_share = treated_n / len(data)
print(f"Ground truth ATTE is {true_atte:.6f}")
print(f"Treated share is {treated_share:.4%} ({treated_n} / {len(data)})")

Ground truth ATTE is 10.914991
Treated share is 4.9490% (4949 / 100000)


In [3]:
from causalis.data_contracts import CausalData

confounders = [
    "tenure_months",
    "avg_sessions_week",
    "spend_last_month",
    "age_years",
    "income_monthly",
    "prior_purchases_12m",
    "support_tickets_90d",
    "premium_user",
    "mobile_user",
    "urban_resident",
    "referred_user",
]

causaldata = CausalData(
    df=data,
    treatment="d",
    outcome="y",
    confounders=confounders,
)
causaldata

CausalData(df=(100000, 13), treatment='d', outcome='y', confounders=['tenure_months', 'avg_sessions_week', 'spend_last_month', 'age_years', 'income_monthly', 'prior_purchases_12m', 'support_tickets_90d', 'premium_user', 'mobile_user', 'urban_resident', 'referred_user'])

# Comparison of Inference

## Causalis

In [4]:
import time
import tracemalloc

import doubleml as dml
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin

from causalis.scenarios.unconfoundedness import IRM

BENCHMARK_SEED = 123
STABILITY_SEEDS = [1, 2, 3, 4, 5]
N_FOLDS = 3
TRIMMING_THRESHOLD = 0.01
CATBOOST_PARAMS = {
    "iterations": 500,
    "depth": 6,
    "learning_rate": 0.1,
    "allow_writing_files": False,
}

data_dml_base = dml.DoubleMLData(
    data,
    y_col="y",
    d_cols="d",
    x_cols=confounders,
)


class SkCatBoostRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, **params):
        self.params = params
        self.model_ = CatBoostRegressor(**params)

    def fit(self, X, y, **fit_params):
        self.model_ = CatBoostRegressor(**self.params)
        self.model_.fit(X, y, verbose=False, **fit_params)
        try:
            self.n_features_in_ = X.shape[1]
        except Exception:
            pass
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def get_params(self, deep=True):
        return dict(self.params)

    def set_params(self, **params):
        self.params.update(params)
        self.model_ = CatBoostRegressor(**self.params)
        return self


class SkCatBoostClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, **params):
        self.params = params
        self.model_ = CatBoostClassifier(**params)

    def fit(self, X, y, **fit_params):
        self.model_ = CatBoostClassifier(**self.params)
        self.model_.fit(X, y, verbose=False, **fit_params)
        if hasattr(self.model_, "classes_"):
            self.classes_ = self.model_.classes_
        else:
            self.classes_ = np.unique(y)
        try:
            self.n_features_in_ = X.shape[1]
        except Exception:
            pass
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        proba = self.model_.predict_proba(X)
        if hasattr(self.model_, "classes_") and list(self.model_.classes_) != list(self.classes_):
            order = [list(self.model_.classes_).index(c) for c in self.classes_]
            proba = np.asarray(proba)[:, order]
        return proba

    def get_params(self, deep=True):
        return dict(self.params)

    def set_params(self, **params):
        self.params.update(params)
        self.model_ = CatBoostClassifier(**self.params)
        return self


def make_learners(seed):
    params = {**CATBOOST_PARAMS, "random_seed": seed}
    return SkCatBoostRegressor(**params), SkCatBoostClassifier(**params)


def measure_call(fn):
    tracemalloc.start()
    t0 = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, elapsed, peak / 1024 / 1024


def extract_causalis_value(result):
    return float(str(result.summary().loc["value", "value"]).split(" ")[0])


def run_causalis(seed):
    np.random.seed(seed)
    ml_g, ml_m = make_learners(seed)

    def fit_and_estimate():
        model = IRM(
            causaldata,
            ml_g=ml_g,
            ml_m=ml_m,
            n_folds=N_FOLDS,
            trimming_threshold=TRIMMING_THRESHOLD,
            random_state=seed,
        ).fit()
        result = model.estimate(score="ATTE")
        return model, result

    (model, result), elapsed, peak = measure_call(fit_and_estimate)
    return {
        "seed": seed,
        "model": model,
        "result": result,
        "atte": extract_causalis_value(result),
        "time_s": elapsed,
        "peak_memory_mb": peak,
    }


def run_doubleml(seed):
    np.random.seed(seed)
    ml_g, ml_m = make_learners(seed)

    def fit_and_estimate():
        model = dml.DoubleMLIRM(
            data_dml_base,
            ml_g=ml_g,
            ml_m=ml_m,
            trimming_threshold=TRIMMING_THRESHOLD,
            n_folds=N_FOLDS,
            score="ATTE",
            draw_sample_splitting=False,
        )
        np.random.seed(seed)
        model.draw_sample_splitting()
        model.fit()
        return model

    model, elapsed, peak = measure_call(fit_and_estimate)
    return {
        "seed": seed,
        "model": model,
        "result": model,
        "atte": float(model.summary.loc["d", "coef"]),
        "time_s": elapsed,
        "peak_memory_mb": peak,
    }


def make_comparison_row(label, run):
    return {
        "library": label,
        "seed": run["seed"],
        "oracle ATTE": round(true_atte, 4),
        "value (ATTE)": round(run["atte"], 4),
        "abs error vs oracle": round(abs(run["atte"] - true_atte), 4),
        "time (s)": round(run["time_s"], 1),
        "peak memory (MB)": round(run["peak_memory_mb"], 1),
    }

In [5]:
causalis_run = run_causalis(BENCHMARK_SEED)
result_causalis = causalis_run["result"]

print(
    f"Causalis matched run (seed={BENCHMARK_SEED}): "
    f"ATTE={causalis_run['atte']:.6f} | "
    f"time={causalis_run['time_s']:.1f}s | "
    f"peak memory={causalis_run['peak_memory_mb']:.1f} MB"
)
result_causalis.summary()

Causalis matched run (seed=123): ATTE=12.331200 | time=93.0s | peak memory=53.8 MB


,value
field,
estimand,ATTE
model,IRM
value,"12.3312 (ci_abs: 7.8782, 16.7842)"
value_relative,"26.7054 (ci_rel: 16.9239, 36.4870)"
alpha,0.0500
p_value,0.0000
is_significant,True
n_treated,4949
n_control,95051


## DoubleML

In [6]:
doubleml_run = run_doubleml(BENCHMARK_SEED)
result_doubleml = doubleml_run["model"]

print(
    f"DoubleML matched run (seed={BENCHMARK_SEED}): "
    f"ATTE={doubleml_run['atte']:.6f} | "
    f"time={doubleml_run['time_s']:.1f}s | "
    f"peak memory={doubleml_run['peak_memory_mb']:.1f} MB"
)
result_doubleml.summary

DoubleML matched run (seed=123): ATTE=12.331221 | time=49.9s | peak memory=40.3 MB


,coef,std err,t,P>|t|,2.5 %,97.5 %
d,12.331221,2.271978,5.427526,5.714047e-08,7.878225,16.784216


In [7]:
pd.Series(
    {
        "benchmark_seed": BENCHMARK_SEED,
        "stability_seeds": STABILITY_SEEDS,
        "n_folds": N_FOLDS,
        "trimming_threshold": TRIMMING_THRESHOLD,
        "catboost_params": CATBOOST_PARAMS,
    },
    name="matched_benchmark_config",
)

benchmark_seed                                                      123
stability_seeds                                         [1, 2, 3, 4, 5]
n_folds                                                               3
trimming_threshold                                                 0.01
catboost_params       {'iterations': 500, 'depth': 6, 'learning_rate...
Name: matched_benchmark_config, dtype: object

In [8]:
matched_comparison = pd.DataFrame(
    [
        make_comparison_row("Causalis", causalis_run),
        make_comparison_row("DoubleML", doubleml_run),
    ]
).set_index("library")
matched_comparison

,seed,oracle ATTE,value (ATTE),abs error vs oracle,time (s),peak memory (MB)
library,,,,,,
Causalis,123,10.915,12.3312,1.4162,93.0,53.8
DoubleML,123,10.915,12.3312,1.4162,49.9,40.3


In [9]:
stability_rows = []
for seed in STABILITY_SEEDS:
    causalis_seed_run = run_causalis(seed)
    doubleml_seed_run = run_doubleml(seed)
    stability_rows.append(
        {
            "seed": seed,
            "causalis_atte": causalis_seed_run["atte"],
            "doubleml_atte": doubleml_seed_run["atte"],
            "gap": causalis_seed_run["atte"] - doubleml_seed_run["atte"],
            "causalis_abs_error": abs(causalis_seed_run["atte"] - true_atte),
            "doubleml_abs_error": abs(doubleml_seed_run["atte"] - true_atte),
            "causalis_time_s": causalis_seed_run["time_s"],
            "doubleml_time_s": doubleml_seed_run["time_s"],
            "causalis_peak_memory_mb": causalis_seed_run["peak_memory_mb"],
            "doubleml_peak_memory_mb": doubleml_seed_run["peak_memory_mb"],
        }
    )

stability_results = pd.DataFrame(stability_rows).round(6)
stability_results

,seed,causalis_atte,doubleml_atte,gap,causalis_abs_error,doubleml_abs_error,causalis_time_s,doubleml_time_s,causalis_peak_memory_mb,doubleml_peak_memory_mb
0,1,12.0184,12.018400,0.000000,1.103409,1.103408,41.884077,38.518782,52.933817,40.233231
1,2,10.6533,10.653295,0.000005,0.261691,0.261696,41.097836,34.594209,52.924817,40.230717
2,3,12.5838,12.583832,-0.000032,1.668809,1.668841,36.624745,35.670716,52.924624,40.230673
3,4,12.5729,12.572861,0.000039,1.657909,1.657870,35.710764,34.821244,52.924680,40.230070
4,5,11.5216,11.521633,-0.000033,0.606609,0.606641,37.362521,34.608914,52.924766,40.229713


# Conclusion

In [10]:
stability_summary = pd.DataFrame(
    [
        {
            "library": "Causalis",
            "mean ATTE": stability_results["causalis_atte"].mean(),
            "std ATTE": stability_results["causalis_atte"].std(ddof=1),
            "mean abs error": stability_results["causalis_abs_error"].mean(),
            "std abs error": stability_results["causalis_abs_error"].std(ddof=1),
            "mean time (s)": stability_results["causalis_time_s"].mean(),
            "std time (s)": stability_results["causalis_time_s"].std(ddof=1),
            "mean peak memory (MB)": stability_results["causalis_peak_memory_mb"].mean(),
            "std peak memory (MB)": stability_results["causalis_peak_memory_mb"].std(ddof=1),
        },
        {
            "library": "DoubleML",
            "mean ATTE": stability_results["doubleml_atte"].mean(),
            "std ATTE": stability_results["doubleml_atte"].std(ddof=1),
            "mean abs error": stability_results["doubleml_abs_error"].mean(),
            "std abs error": stability_results["doubleml_abs_error"].std(ddof=1),
            "mean time (s)": stability_results["doubleml_time_s"].mean(),
            "std time (s)": stability_results["doubleml_time_s"].std(ddof=1),
            "mean peak memory (MB)": stability_results["doubleml_peak_memory_mb"].mean(),
            "std peak memory (MB)": stability_results["doubleml_peak_memory_mb"].std(ddof=1),
        },
    ]
).round(4).set_index("library")
stability_summary

,mean ATTE,std ATTE,mean abs error,std abs error,mean time (s),std time (s),mean peak memory (MB),std peak memory (MB)
library,,,,,,,,
Causalis,11.87,0.8105,1.0597,0.6271,38.5360,2.7742,52.9265,0.0041
DoubleML,11.87,0.8105,1.0597,0.6271,35.6428,1.6670,40.2309,0.0014


In [11]:
agreement_check = pd.Series(
    {
        "oracle_atte": round(true_atte, 6),
        "treated_share": round(treated_share, 6),
        "matched_seed_gap": round(abs(causalis_run["atte"] - doubleml_run["atte"]), 6),
        "max_gap_over_stability_seeds": round(stability_results["gap"].abs().max(), 6),
        "near_identical_for_matched_seed": abs(causalis_run["atte"] - doubleml_run["atte"]) < 0.1,
        "identical_over_stability_seeds": bool((stability_results["gap"].abs() < 1e-9).all()),
    },
    name="agreement_check",
)
agreement_check

oracle_atte                        10.914991
treated_share                        0.04949
matched_seed_gap                    0.000021
max_gap_over_stability_seeds        0.000039
near_identical_for_matched_seed         True
identical_over_stability_seeds         False
Name: agreement_check, dtype: object

With matched learners, fold count, trimming, and seeded sample splitting, Causalis and DoubleML agree up to numerical precision. The remaining movement is seed-to-seed benchmark variability, not a difference in the ATTE estimand.